# Monthly Passenger Demand Feature Sets

This notebook prepares four candidate feature sets for forecasting monthly passenger demand (`OD_PAX`) by directional origin–destination route. It engineers features only; model training and evaluation are left for a later step.

Place `AA_PAX_FORECASTING.csv` and `AA_AIRPORT_INFO.csv` in the `team-06-modeling/` folder. The code searches from either the repository root or this notebook's folder.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

TARGET = "OD_PAX"
ORIGIN = "ORG_AIRPRT"
DESTINATION = "DES_AIRPRT"
ROUTE_COLUMNS = [ORIGIN, DESTINATION]
MONTH = "MONTH_START"
MONTH_INDEX = "MONTH_INDEX"

data_dir_candidates = [
    Path.cwd() / "team-06-modeling",
    Path.cwd(),
    Path.cwd().parent,
]
DATA_DIR = next(
    (
        path
        for path in data_dir_candidates
        if (path / "AA_PAX_FORECASTING.csv").is_file()
        and (path / "AA_AIRPORT_INFO.csv").is_file()
    ),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find both CSV files. Place them in team-06-modeling/ "
        "or update DATA_DIR to their folder."
    )

passengers = pd.read_csv(DATA_DIR / "AA_PAX_FORECASTING.csv", index_col=0)
airports = pd.read_csv(DATA_DIR / "AA_AIRPORT_INFO.csv", index_col=0)

required_passenger_columns = {
    "DEP_MONTH", "DEP_YEAR", ORIGIN, DESTINATION, TARGET
}
required_airport_columns = {
    "AIRPRT_CD", "CNTRY_CD", "STATE_PROVNC_CD", "CITY_METRO_IATA_CD",
    "LAT_DEGREE_QTY", "LAT_MINUTE_QTY", "LAT_SECOND_QTY",
    "LAT_HEMSPHR_CD", "LNGTD_DEGREE_QTY", "LNGTD_MINUTE_QTY",
    "LNGTD_SECOND_QTY", "LNGTD_HEMSPHR_CD", "LNGST_RUNWAY_FT_QTY",
    "ELEVATN_FT_QTY",
}
missing_passenger_columns = required_passenger_columns - set(passengers.columns)
missing_airport_columns = required_airport_columns - set(airports.columns)
if missing_passenger_columns:
    raise ValueError(f"Passenger CSV is missing columns: {sorted(missing_passenger_columns)}")
if missing_airport_columns:
    raise ValueError(f"Airport CSV is missing columns: {sorted(missing_airport_columns)}")

passengers["DEP_MONTH"] = pd.to_numeric(passengers["DEP_MONTH"], errors="raise").astype("int8")
passengers["DEP_YEAR"] = pd.to_numeric(passengers["DEP_YEAR"], errors="raise").astype("int16")
passengers[TARGET] = pd.to_numeric(passengers[TARGET], errors="raise")
if passengers[["DEP_MONTH", "DEP_YEAR", ORIGIN, DESTINATION, TARGET]].isna().any().any():
    raise ValueError("Required route, date, or target fields contain missing values.")
if not passengers["DEP_MONTH"].between(1, 12).all():
    raise ValueError("DEP_MONTH must be in the range 1 through 12.")

passengers[MONTH] = pd.to_datetime(
    dict(year=passengers["DEP_YEAR"], month=passengers["DEP_MONTH"], day=1)
)
passengers[MONTH_INDEX] = passengers["DEP_YEAR"] * 12 + passengers["DEP_MONTH"] - 1
if passengers.duplicated(ROUTE_COLUMNS + [MONTH]).any():
    raise ValueError("Expected at most one passenger record per route and month.")

print(f"Passenger records: {len(passengers):,}")
print(f"Date range: {passengers[MONTH].min():%Y-%m} to {passengers[MONTH].max():%Y-%m}")
print(f"Directional routes: {passengers[ROUTE_COLUMNS].drop_duplicates().shape[0]:,}")
print(f"Airport reference records: {len(airports):,}")

## Feature engineering

Route lags use exact calendar-month matches, so a missing route-month stays missing rather than being filled with zero or treated as the previous month. Trailing demand statistics use prior observations in rolling calendar windows and exclude the target month. Airport totals are calculated from observed records only, so they may not represent complete airport-wide traffic if route-month records are missing.

In [ ]:
def add_exact_route_lag(frame: pd.DataFrame, lag_months: int) -> pd.DataFrame:
    """Add a route passenger lag for an exact prior calendar month."""
    lag_name = f"route_pax_lag_{lag_months}m"
    lookup = frame[ROUTE_COLUMNS + [MONTH_INDEX, TARGET]].copy()
    lookup[MONTH_INDEX] = lookup[MONTH_INDEX] + lag_months
    lookup = lookup.rename(columns={TARGET: lag_name})
    return frame.merge(
        lookup,
        on=ROUTE_COLUMNS + [MONTH_INDEX],
        how="left",
        validate="one_to_one",
        sort=False,
    )


def add_trailing_route_stats(
    frame: pd.DataFrame, window: str, suffix: str
) -> pd.DataFrame:
    """Add prior-only route mean and count over a calendar-time window."""
    stats = (
        frame.groupby(ROUTE_COLUMNS, sort=False)
        .rolling(window=window, on=MONTH, closed="left")[TARGET]
        .agg(["mean", "count"])
        .reset_index()
        .rename(
            columns={
                "mean": f"route_pax_mean_{suffix}",
                "count": f"route_observed_months_{suffix}",
            }
        )
    )
    return frame.merge(
        stats,
        on=ROUTE_COLUMNS + [MONTH],
        how="left",
        validate="one_to_one",
        sort=False,
    )


def add_airport_history(
    frame: pd.DataFrame, airport_column: str, role: str
) -> pd.DataFrame:
    """Add previous-month traffic and trailing route-record frequency by airport."""
    airport_month = (
        frame.groupby([airport_column, MONTH, MONTH_INDEX], as_index=False, sort=False)
        .agg(
            airport_observed_pax=(TARGET, "sum"),
            airport_route_records=(TARGET, "size"),
        )
        .rename(columns={airport_column: "AIRPORT_KEY"})
        .sort_values(["AIRPORT_KEY", MONTH])
    )

    lag_lookup = airport_month[["AIRPORT_KEY", MONTH_INDEX, "airport_observed_pax"]].copy()
    lag_lookup[MONTH_INDEX] = lag_lookup[MONTH_INDEX] + 1
    lag_lookup = lag_lookup.rename(
        columns={"airport_observed_pax": f"{role}_pax_previous_month"}
    )

    rolling = (
        airport_month.groupby("AIRPORT_KEY", sort=False)
        .rolling(window="366D", on=MONTH, closed="left")["airport_route_records"]
        .sum()
        .reset_index()
        .rename(columns={"airport_route_records": f"{role}_route_frequency_12m"})
    )
    airport_features = airport_month.merge(
        lag_lookup,
        on=["AIRPORT_KEY", MONTH_INDEX],
        how="left",
        validate="one_to_one",
    ).merge(
        rolling,
        on=["AIRPORT_KEY", MONTH],
        how="left",
        validate="one_to_one",
    )

    lookup = airport_features[
        [
            "AIRPORT_KEY", MONTH, f"{role}_pax_previous_month",
            f"{role}_route_frequency_12m",
        ]
    ].rename(columns={"AIRPORT_KEY": airport_column})
    return frame.merge(
        lookup,
        on=[airport_column, MONTH],
        how="left",
        validate="many_to_one",
        sort=False,
    )


def decimal_coordinate(degrees, minutes, seconds, hemisphere):
    """Convert degree/minute/second coordinates and hemisphere to decimal degrees."""
    value = (
        pd.to_numeric(degrees, errors="coerce")
        + pd.to_numeric(minutes, errors="coerce") / 60
        + pd.to_numeric(seconds, errors="coerce") / 3600
    )
    sign = pd.Series(hemisphere, index=value.index).astype("string").str.upper().map(
        {"N": 1, "E": 1, "S": -1, "W": -1}
    )
    return value * sign


def haversine_km(lat1, lon1, lat2, lon2):
    """Calculate great-circle airport distance in kilometers."""
    lat1, lon1, lat2, lon2 = [np.radians(value) for value in (lat1, lon1, lat2, lon2)]
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371.0088 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

## Build the feature sets

The feature-set definitions are lists of predictor columns, with `OD_PAX` kept separately as the target. `get_feature_frame()` materializes only the selected set, avoiding four large copies of this dataset in memory.

In [ ]:
features = passengers.sort_values(ROUTE_COLUMNS + [MONTH]).copy()
features["DEP_MONTH_SIN"] = np.sin(2 * np.pi * features["DEP_MONTH"] / 12)
features["DEP_MONTH_COS"] = np.cos(2 * np.pi * features["DEP_MONTH"] / 12)
features["TIME_INDEX"] = features[MONTH_INDEX] - features[MONTH_INDEX].min()
features["ROUTE_ID"] = features[ORIGIN].astype("string") + "->" + features[DESTINATION].astype("string")

# Historical route demand and historical route record frequency.
features = add_exact_route_lag(features, 1)
features = add_exact_route_lag(features, 12)
for window, suffix in [("92D", "3m"), ("184D", "6m"), ("366D", "12m")]:
    features = add_trailing_route_stats(features, window, suffix)
features["route_frequency_12m"] = features["route_observed_months_12m"]

# Historical airport activity: outbound at origin, inbound at destination.
features = add_airport_history(features, ORIGIN, "origin")
features = add_airport_history(features, DESTINATION, "destination")

# Convert airport coordinates and attach origin/destination context.
airport_features = airports.copy()
airport_features["LATITUDE"] = decimal_coordinate(
    airport_features["LAT_DEGREE_QTY"],
    airport_features["LAT_MINUTE_QTY"],
    airport_features["LAT_SECOND_QTY"],
    airport_features["LAT_HEMSPHR_CD"],
)
airport_features["LONGITUDE"] = decimal_coordinate(
    airport_features["LNGTD_DEGREE_QTY"],
    airport_features["LNGTD_MINUTE_QTY"],
    airport_features["LNGTD_SECOND_QTY"],
    airport_features["LNGTD_HEMSPHR_CD"],
)
airport_attributes = [
    "AIRPRT_CD", "CNTRY_CD", "STATE_PROVNC_CD", "CITY_METRO_IATA_CD",
    "LATITUDE", "LONGITUDE", "LNGST_RUNWAY_FT_QTY", "ELEVATN_FT_QTY",
]
airport_lookup = airport_features[airport_attributes].drop_duplicates("AIRPRT_CD")

for role, code_column in [("origin", ORIGIN), ("destination", DESTINATION)]:
    role_lookup = airport_lookup.rename(
        columns={column: f"{role}_{column}" for column in airport_attributes if column != "AIRPRT_CD"}
    )
    features = features.merge(
        role_lookup,
        left_on=code_column,
        right_on="AIRPRT_CD",
        how="left",
        validate="many_to_one",
        sort=False,
    ).drop(columns="AIRPRT_CD")

features["DISTANCE_KM"] = haversine_km(
    features["origin_LATITUDE"],
    features["origin_LONGITUDE"],
    features["destination_LATITUDE"],
    features["destination_LONGITUDE"],
)

calendar_features = [
    ORIGIN, DESTINATION, "ROUTE_ID", "DEP_MONTH", "DEP_YEAR",
    "DEP_MONTH_SIN", "DEP_MONTH_COS", "TIME_INDEX",
]
historical_features = [
    "route_pax_lag_1m", "route_pax_lag_12m",
    "route_pax_mean_3m", "route_pax_mean_6m", "route_pax_mean_12m",
    "route_observed_months_3m", "route_observed_months_6m",
    "route_observed_months_12m", "route_frequency_12m",
    "origin_pax_previous_month", "destination_pax_previous_month",
    "origin_route_frequency_12m", "destination_route_frequency_12m",
]
geographic_features = [
    "origin_CNTRY_CD", "destination_CNTRY_CD",
    "origin_STATE_PROVNC_CD", "destination_STATE_PROVNC_CD",
    "origin_CITY_METRO_IATA_CD", "destination_CITY_METRO_IATA_CD",
    "origin_LATITUDE", "origin_LONGITUDE",
    "destination_LATITUDE", "destination_LONGITUDE", "DISTANCE_KM",
    "origin_LNGST_RUNWAY_FT_QTY", "destination_LNGST_RUNWAY_FT_QTY",
    "origin_ELEVATN_FT_QTY", "destination_ELEVATN_FT_QTY",
]

FEATURE_SETS = {
    "1A_route_calendar": calendar_features,
    "1B_route_calendar_history": calendar_features + historical_features,
    "2_route_calendar_geography": calendar_features + geographic_features,
    "3_combined": calendar_features + historical_features + geographic_features,
}

def get_feature_frame(feature_set_name: str) -> pd.DataFrame:
    """Return one selected feature set together with the passenger target."""
    if feature_set_name not in FEATURE_SETS:
        raise KeyError(f"Unknown feature set {feature_set_name!r}; choose from {list(FEATURE_SETS)}")
    return features[FEATURE_SETS[feature_set_name] + [TARGET]].copy()

for name, columns in FEATURE_SETS.items():
    print(f"{name}: {len(columns)} predictors")

print(f"Airport metadata missing for origin: {features['origin_LATITUDE'].isna().mean():.1%}")
print(f"Airport metadata missing for destination: {features['destination_LATITUDE'].isna().mean():.1%}")
print("Feature tables are ready; for example, use get_feature_frame('1A_route_calendar').")

## What each set contains and why

- **Feature Set 1A — Route, date, seasonality:** directional origin and destination codes, a route identifier, month and year, cyclic month encodings, and a time index. This is the baseline for recurring seasonality and route-level differences.
- **Feature Set 1B — Historical demand:** all 1A predictors plus exact prior-month and prior-year route passenger counts, rolling route means and observation counts, route appearance frequency, and previous-month airport passenger totals and route-record frequencies. These summarize demand history without treating an absent route-month as zero.
- **Feature Set 2 — Geography:** all 1A predictors plus origin/destination country, state/province, metro area, decimal coordinates, longest runway, elevation, and great-circle distance in kilometers. Coordinates and airport attributes are candidates to validate, not guaranteed improvements.
- **Feature Set 3 — Combined:** all 1B and Feature Set 2 predictors, so you can test whether geography adds value beyond historical demand.

### Notes before modeling

- `OD_PAX` is the target and is included only as the final column returned by `get_feature_frame`; it is not in `FEATURE_SETS`.
- `OD_REV_USD`, `SF_LOAD_TMS`, and the saved CSV row index are excluded as predictors. Same-month revenue could leak information about the target, and load time is not a departure-time demand signal.
- Treat airport and route codes as categorical inputs in the eventual model; do not interpret their codes as numeric magnitudes.
- Missing lag/rolling values mean prior route data were unavailable. Do not fill them with zero without a justified policy.
- These historical features are valid for one-step-ahead predictions when previous actual months are known. At a multi-month forecast horizon, future actual passenger counts are unavailable: use direct horizon-specific forecasts or recursively feed model predictions, never actual future `OD_PAX` values.
- `OD_PAX` measures passengers represented in this dataset, not necessarily unconstrained demand; capacity and schedule data are not supplied.
- Compare feature sets with identical chronological validation periods and horizons against a same-route/same-month-last-year baseline. Evaluate MAE, WAPE, and forecast bias; decide feature value from held-out performance, not feature importance alone.